**NOTE** This notebook contains `pyecharts` interactive charts. Unfortunately, they don't render in the GitHub notebook preview, so you'll need to view the notebook locally.

In [1]:
from sklearn import datasets, metrics
from sklearn.linear_model import LogisticRegression
import numpy as np
import pandas as pd

from thds.mllegos.eval import cls_report

In [2]:
digits_dataset  = datasets.load_digits(as_frame=True)

n_samples = len(digits_dataset.images)
features = digits_dataset.images.reshape((n_samples, -1))

labels = digits_dataset.target
label_names = digits_dataset.target_names

In [3]:
# All of this context info is basically meaningless, but you can play around with using the various
# columns to color the points in the scatter plot
# Notes:
#   - only int, string, and boolean columns can be used for coloring
#   - classes with no entries in context_info will still be included in the plot

context_info = pd.DataFrame(
    {
        'name': ['zero', 'one', 'two', 'three', 'four', 'five', 'six'],
        'count': [0,1,2,3,4,5,6],
        'is_even': [None, False, True, False, True, False, True],
        'scale': [0.0, 1.1, 2.2, 3.3, 4.4, 5.5, 6.6],
    },
    index = label_names[:7],
)

In [4]:
clf = LogisticRegression(penalty="l2", max_iter=1)
clf.fit(features, labels)

/Users/cailey.kerley/Projects/ds-monorepo/libs/mllegos/.venv/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression(max_iter=1)

In [5]:
report = metrics.classification_report(
    labels,
    clf.predict(features),
    target_names=label_names,
    output_dict=True,
    zero_division=np.nan,
)

## Dict -> DataFrame

In [6]:
cls_report.to_pandas(report, context_info=context_info)

,precision,recall,f1-score,support,name,count,is_even,scale
0,0.988764,0.988764,0.988764,178.0,zero,0.0,None,0.0
1,0.673554,0.895604,0.768868,182.0,one,1.0,False,1.1
2,0.980769,0.864407,0.918919,177.0,two,2.0,True,2.2
3,0.824390,0.923497,0.871134,183.0,three,3.0,False,3.3
4,0.948571,0.917127,0.932584,181.0,four,4.0,True,4.4
5,0.946108,0.868132,0.905444,182.0,five,5.0,False,5.5
6,0.921466,0.972376,0.946237,181.0,six,6.0,True,6.6
7,0.918919,0.949721,0.934066,179.0,NaN,NaN,NaN,NaN
8,0.884615,0.660920,0.756579,174.0,NaN,NaN,NaN,NaN
9,0.845238,0.788889,0.816092,180.0,NaN,NaN,NaN,NaN


## Interactive Scatterplots!

### Precision vs Recall

In [7]:
cls_report.multiclass_performance_viz(
    report,
    'recall',
    'precision',
).render_notebook()

### Precision vs Recall colored by class label

In [8]:
cls_report.multiclass_performance_viz(
    report,
    'recall',
    'precision',
    color_by='class'
).render_notebook()

### F1 Score vs Support colored by `is_even`

In [9]:
chart = cls_report.multiclass_performance_viz(
    report,
    'support',
    'f1-score',
    context_info=context_info,
    color_by='count'
)

chart.render_notebook()

### Now save it to an html so I can share it

In [10]:
chart.render("f1-score_vs_support.html")

'/Users/cailey.kerley/Projects/ds-monorepo/libs/mllegos/notebooks/f1-score_vs_support.html'

## Slight Advanced: Custom Scatterplots & Bar Charts

Want to make a `pyecharts` chart but don't really feel like learning how to use `pyecharts` directly? You can also create some basic
scatterplot or bar charts using `mllegos.eval.viz.basic`

In [11]:
from thds.mllegos.eval.viz import basic

In [12]:
iris_dataset = datasets.load_iris(as_frame=True)
iris_Xy = iris_dataset.data
iris_Xy['class'] = iris_dataset.target_names[iris_dataset.target]
iris_Xy.columns

Index(['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)',
       'petal width (cm)', 'class'],
      dtype='object')

### Prep data
`pyecharts` only works with native python types, so you have to convert pandas dataframes & numpy arrays to native lists

In [13]:
data_table = list(iris_Xy.itertuples(index=False, name=None))
table_columns = iris_Xy.columns.tolist()
class_names = iris_dataset.target_names.tolist()

### Make a Scatterplot of Iris Sample Features

In [18]:
basic.scatterplot(
    data = data_table,
    field_names = table_columns,
    x_field = 'sepal length (cm)',
    y_field = 'sepal width (cm)',
    tooltip = table_columns,
    color_values = class_names,
    color_field = 'class',
    title="Sepal Width vs Sepal Length for Iris Samples",
).render_notebook()

### Make a Barchart of Mean Iris Petal Length

In [15]:
iris_mean_measurements = iris_Xy.groupby('class').mean().round(4).reset_index()

In [16]:
mm_data_table = list(iris_mean_measurements.itertuples(index=False, name=None))
mm_table_columns = iris_mean_measurements.columns.tolist()
mm_class_names = iris_mean_measurements['class'].tolist()
mm_tooltip = iris_mean_measurements.drop(columns=['class']).columns.tolist()

In [17]:
basic.barchart(
    data = mm_data_table,
    field_names = mm_table_columns,
    numeric_field = 'petal length (cm)',
    categorical_field='class',
    tooltip=mm_tooltip,
    color_values = mm_class_names,
    color_field='class',
    zoom_end=None, # only 3 bars so we don't need to zoom
    title="Mean Measurements for Each Iris Species",
).render_notebook()